In [1]:
import os
import ctypes
import torch

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = os.path.expanduser("~/.local/lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")

LIB = os.path.expanduser("~/.local/lib/libespeak-ng.so")
ctypes.CDLL(LIB)

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
device = torch.device("cuda")

In [2]:
!espeak-ng --version

eSpeak NG text-to-speech: 1.52.0  Data at: /home/minh_duc/.local/share/espeak-ng-data


In [3]:
from neutts import NeuTTS
import soundfile as sf

tts = NeuTTS(
   backbone_repo="neuphonic/neutts-nano", # or 'neutts-nano-q4-gguf' with llama-cpp-python installed
   backbone_device="cuda",
   codec_repo="neuphonic/neucodec",
   codec_device="cuda"
)
input_text = "Twas brillig, and the slithy toves Did gyre and gimble in the wabe; All mimsy were the borogoves, And the mome raths outgrabe."

ref_text = "samples/jo.txt"
ref_audio_path = "samples/jo.wav"

ref_text = open(ref_text, "r").read().strip()
ref_codes = tts.encode_reference(ref_audio_path)

wav, codes = tts.infer(input_text, ref_codes, ref_text, return_codes=True)
sf.write("test.wav", wav, 24000)

/home/minh_duc/miniconda3/envs/neutts_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano on cuda ...
Loading codec from: neuphonic/neucodec on cuda ...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 14169.95it/s]
/home/minh_duc/miniconda3/envs/neutts_env/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/minh_duc/miniconda3/envs/neutts_env/lib/python3.10/site-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


loaded PerthNet (Implicit) at step 250,000


In [4]:
codes

'<|speech_22934|><|speech_21862|><|speech_6210|><|speech_5478|><|speech_52406|><|speech_29757|><|speech_13598|><|speech_32010|><|speech_56489|><|speech_57266|><|speech_34808|><|speech_1908|><|speech_2412|><|speech_938|><|speech_35660|><|speech_21470|><|speech_35788|><|speech_926|><|speech_1484|><|speech_33373|><|speech_18316|><|speech_357|><|speech_1856|><|speech_23378|><|speech_48662|><|speech_47642|><|speech_60776|><|speech_55769|><|speech_55768|><|speech_52449|><|speech_55562|><|speech_36538|><|speech_36160|><|speech_63182|><|speech_33040|><|speech_16397|><|speech_59960|><|speech_43694|><|speech_36213|><|speech_1037|><|speech_32788|><|speech_45404|><|speech_36904|><|speech_383|><|speech_26995|><|speech_57498|><|speech_53929|><|speech_54774|><|speech_50657|><|speech_50421|><|speech_50420|><|speech_45270|><|speech_18028|><|speech_570|><|speech_444|><|speech_22|><|speech_17557|><|speech_18851|><|speech_26342|><|speech_62947|><|speech_34370|><|speech_13857|><|speech_7044|><|speech_60946

In [5]:
wav.shape

(156480,)

In [6]:
code = tts.encode_reference(ref_audio_path="./test.wav")

In [7]:
print(code)
code.shape

tensor([17751,  6246,  5207,  6310, 56501, 13629, 13326, 48398, 55482, 53169,
        51132,   949,  2332,   682, 19276, 21450, 36812,   926,   716, 49773,
         2956,   357,  2880, 27201, 32026, 63769, 59736, 56792, 51668, 52417,
        56699, 36420, 35294, 62976, 33037, 33848, 42589, 61044, 18762, 16420,
        16716, 61736, 20797,   305, 26087, 54365, 54010, 50609, 51426, 33008,
        50416, 45286,   684, 16954,   444, 16390, 17558, 17907, 42726, 63059,
        34369, 29281,  2949, 60967, 56609, 18961, 56341, 53505, 54634, 53178,
        56934,  1546,  8737, 10013,  1810, 51606, 52449, 54561, 30562, 35647,
        34606, 17723, 54902, 50418, 52402, 59449, 21790, 58709, 55142, 36573,
        59001, 18797, 34100, 47742, 17716,  1189, 18804,  7611, 35670, 65063,
        65323, 60726, 61282, 52658, 36018, 54826, 33650, 13921,   634, 46870,
        63155, 65458, 65267, 56695, 55615, 45371, 45953, 42692, 30025, 15564,
        28806, 29920, 29057, 29153, 47681, 30145, 28162,  5829, 

torch.Size([326])

In [8]:
def speech_ids_to_codes(speech_ids):
    # Handle torch tensor
    if hasattr(speech_ids, "tolist"):
        speech_ids = speech_ids.tolist()

    return "".join(f"<|speech_{i}|>" for i in speech_ids)


In [9]:
code_str = speech_ids_to_codes(code)

In [10]:
len(code_str)

5159

In [11]:
code.shape

torch.Size([326])

In [12]:
de_wav = tts._decode(code_str)

In [13]:
code_str

'<|speech_17751|><|speech_6246|><|speech_5207|><|speech_6310|><|speech_56501|><|speech_13629|><|speech_13326|><|speech_48398|><|speech_55482|><|speech_53169|><|speech_51132|><|speech_949|><|speech_2332|><|speech_682|><|speech_19276|><|speech_21450|><|speech_36812|><|speech_926|><|speech_716|><|speech_49773|><|speech_2956|><|speech_357|><|speech_2880|><|speech_27201|><|speech_32026|><|speech_63769|><|speech_59736|><|speech_56792|><|speech_51668|><|speech_52417|><|speech_56699|><|speech_36420|><|speech_35294|><|speech_62976|><|speech_33037|><|speech_33848|><|speech_42589|><|speech_61044|><|speech_18762|><|speech_16420|><|speech_16716|><|speech_61736|><|speech_20797|><|speech_305|><|speech_26087|><|speech_54365|><|speech_54010|><|speech_50609|><|speech_51426|><|speech_33008|><|speech_50416|><|speech_45286|><|speech_684|><|speech_16954|><|speech_444|><|speech_16390|><|speech_17558|><|speech_17907|><|speech_42726|><|speech_63059|><|speech_34369|><|speech_29281|><|speech_2949|><|speech_60967

In [14]:
from IPython.display import Audio, display

display(Audio(data=de_wav, rate=24000))


In [15]:
tts.backbone

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(194256, 576, padding_idx=128001)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=2304, bias=False)
          (up_proj): Linear(in_features=576, out_features=2304, bias=False)
          (down_proj): Linear(in_features=2304, out_features=576, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotar

In [16]:
tts.codec

NeuCodec(
  (semantic_model): Wav2Vec2BertModel(
    (feature_projection): Wav2Vec2BertFeatureProjection(
      (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=160, out_features=1024, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Wav2Vec2BertEncoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): ModuleList(
        (0-23): 24 x Wav2Vec2BertEncoderLayer(
          (ffn1_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (ffn1): Wav2Vec2BertFeedForward(
            (intermediate_dropout): Dropout(p=0.0, inplace=False)
            (intermediate_dense): Linear(in_features=1024, out_features=4096, bias=True)
            (intermediate_act_fn): SiLU()
            (output_dense): Linear(in_features=4096, out_features=1024, bias=True)
            (output_dropout): Dropout(p=0.0, inplace=False)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps